# Real-Time Monitoring

Stream a variational job's energy to a live convergence curve, the same descent CloudWatch shows.

**Objectives:**
- Run a small VQE optimization loop locally and collect the energy each iteration
- Plot the live convergence curve -- exactly what `log_metric` would render in CloudWatch
- Write the job entry-point that calls `log_metric` and applies a `stopping_condition`, gated behind `RUN_ON_AWS`
- Understand the three job channels: hyperparameters, input data, and the metrics stream

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

import numpy as np
import matplotlib.pyplot as plt

from braket.circuits import Circuit, Observable
from braket.devices import LocalSimulator

# These import fine with no AWS credentials -- we only CALL them inside the
# gated `if RUN_ON_AWS:` block far below. Importing != hitting AWS.
from braket.aws import AwsQuantumJob
from braket.jobs.metrics import log_metric
from braket.jobs import save_job_result

np.random.seed(7)

# Local simulator: free, instant, no credentials. Everything executed in this
# notebook runs here.
device = LocalSimulator()
print("LocalSimulator ready -- no AWS credentials required")

## 1. The problem: a job is a black box until it reports

When you hand a 40-iteration VQE to a managed Hybrid Job, the container runs on AWS and you are not watching a `print` loop on your laptop. The only window into *is it converging?* is the metrics the script chooses to emit. Braket's answer is `log_metric`: each call ships one numeric value, tagged with an iteration number, to CloudWatch, where it becomes a live time series.

Before we package any of that, we build and run the exact optimization locally so the curve is real and reproducible. We minimize the energy of a tiny Hamiltonian

$$H = Z_0 Z_1 + \tfrac{1}{2} X_0$$

with a two-qubit ansatz $\mathrm{RY}(\theta_0)\,\mathrm{RY}(\theta_1)\,\mathrm{CNOT}_{0,1}$. The exact ground-state energy is $-\sqrt{1.25} \approx -1.118$, so we have a target to converge toward.

In [ ]:
# The ansatz: one fixed structure, two free angles chosen by the optimizer.
def ansatz(theta0, theta1):
    return Circuit().ry(0, theta0).ry(1, theta1).cnot(0, 1)

# Energy = <H>. The two terms Z0 Z1 and X0 do NOT commute, so they cannot be
# sampled simultaneously with shots. We use exact expectation (shots=0) on the
# LocalSimulator -- deterministic, fast, and credential-free.
def energy(params):
    c = ansatz(params[0], params[1])
    c.expectation(observable=Observable.Z() @ Observable.Z(), target=[0, 1])
    c.expectation(observable=Observable.X(), target=0)
    result = device.run(c, shots=0).result()
    zz, x0 = result.values[0], result.values[1]
    return 1.0 * zz + 0.5 * x0

# Sanity check at the starting point.
start = np.array([2.5, 2.0])
print(f"energy at start params {start}: {energy(start):.4f}")
print("exact ground-state energy: -1.1180  (= -sqrt(1.25))")

## 2. Run the optimization loop and collect the history

This is a plain gradient descent driven by the parameter-shift rule (the exact analytic gradient of a rotation gate is a difference of two energies evaluated at $\theta \pm \pi/2$). Every iteration we append the current energy to a list -- that list **is** the metric stream. In a real job each append is replaced by a `log_metric` call; here we keep it local so the printed history is genuine.

In [ ]:
lr = 0.5            # learning rate (a hyperparameter, in job terms)
max_iter = 40       # iteration budget
shift = np.pi / 2   # parameter-shift offset

params = start.copy()
energy_history = []

for i in range(max_iter):
    e = energy(params)
    energy_history.append(e)  # <-- in a job, this is log_metric(... iteration_number=i)

    # Parameter-shift gradient: exact for single-qubit rotations.
    grad = np.zeros(2)
    for k in range(2):
        plus = params.copy();  plus[k]  += shift
        minus = params.copy(); minus[k] -= shift
        grad[k] = 0.5 * (energy(plus) - energy(minus))

    params = params - lr * grad

print(f"iteration  0: energy = {energy_history[0]:.4f}")
print(f"iteration {max_iter-1}: energy = {energy_history[-1]:.4f}")
print(f"final params: {np.round(params, 3)}")
print(f"target (exact): -1.1180")
assert energy_history[-1] < energy_history[0], "energy must decrease"
print("\nConverged downward toward the ground state.")

## 3. Plot the live convergence curve

This is the picture CloudWatch draws from the `energy` metric: iteration on the x-axis, energy on the y-axis, descending toward the ground state. The dashed line is the exact minimum -- and a natural place to set a `stopping_condition` so the job halts (and stops billing) once it is close enough.

In [ ]:
exact_min = -np.sqrt(1.25)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(len(energy_history)), energy_history,
        marker="o", markersize=4, linewidth=1.6, color="#2563eb",
        label="energy (logged each iteration)")
ax.axhline(exact_min, linestyle="--", color="#64748b",
           label=f"exact ground state = {exact_min:.4f}")
ax.set_xlabel("iteration_number")
ax.set_ylabel("energy")
ax.set_title("VQE convergence -- the curve CloudWatch would stream")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"gap to exact minimum: {energy_history[-1] - exact_min:.5f}")

## 4. The metrics channel, and the two other job channels

The loop above moved one number out of the algorithm each step. A Hybrid Job formalizes three channels around your script:

- **Hyperparameters** -- key/value knobs passed in at create time (`lr`, `max_iter`, shot count). Your script reads them from `os.environ` / the hyperparameters dict, so you can re-run with new settings without editing code.
- **Input data** -- larger artifacts (a molecular geometry, a training set, a graph) staged from S3 into the container at startup, available under a local path.
- **Output artifacts and metrics** -- results written with `save_job_result`, and numeric metrics streamed live via `log_metric(metric_name=..., value=..., iteration_number=...)`.

`log_metric` only works **inside a running job** (it writes to the job's CloudWatch log stream). Called at top level with no job context it has nothing to write to, so we never call it here -- it lives only in the entry-point string below.

## 5. The job entry-point (shown as code, not executed)

Below is the algorithm script Braket would run inside the container. It is the same loop you just ran, with two production additions:

1. `log_metric(metric_name="energy", value=e, iteration_number=i)` replaces the `energy_history.append`.
2. A **stopping condition** -- if the energy stops improving by more than `tol`, the loop breaks early so the instance shuts down promptly (cost discipline).

We hold it as a string and write it to a file. We do **not** run it here, because `log_metric` and `save_job_result` only function inside a job. Writing the file touches no AWS.

In [ ]:
entry_point_source = '''
import os
import numpy as np
from braket.circuits import Circuit, Observable
from braket.devices import LocalSimulator
from braket.jobs import save_job_result
from braket.jobs.metrics import log_metric


def main():
    # --- hyperparameters channel ---
    hp = {k: v for k, v in os.environ.items() if k.startswith("HP_")}
    lr = float(hp.get("HP_lr", 0.5))
    max_iter = int(hp.get("HP_max_iter", 40))
    tol = float(hp.get("HP_tol", 1e-4))

    # Inside a real job, point this at the job's device via
    # os.environ["AMZN_BRAKET_DEVICE_ARN"]; LocalSimulator keeps the demo free.
    device = LocalSimulator()

    def energy(params):
        c = Circuit().ry(0, params[0]).ry(1, params[1]).cnot(0, 1)
        c.expectation(observable=Observable.Z() @ Observable.Z(), target=[0, 1])
        c.expectation(observable=Observable.X(), target=0)
        r = device.run(c, shots=0).result()
        return 1.0 * r.values[0] + 0.5 * r.values[1]

    params = np.array([2.5, 2.0])
    shift = np.pi / 2
    prev = None
    e = None

    for i in range(max_iter):
        e = energy(params)
        # --- metrics channel: this is the live CloudWatch series ---
        log_metric(metric_name="energy", value=float(e), iteration_number=i)

        # --- stopping_condition: halt once converged, stop paying ---
        if prev is not None and abs(prev - e) < tol:
            log_metric(metric_name="stopped_early", value=float(i), iteration_number=i)
            break
        prev = e

        grad = np.zeros(2)
        for k in range(2):
            plus = params.copy();  plus[k]  += shift
            minus = params.copy(); minus[k] -= shift
            grad[k] = 0.5 * (energy(plus) - energy(minus))
        params = params - lr * grad

    # --- output channel ---
    save_job_result({"final_energy": float(e), "optimal_params": params.tolist()})


if __name__ == "__main__":
    main()
'''

with open("vqe_monitored_job.py", "w") as f:
    f.write(entry_point_source)

# Verify it is valid Python without running it (no AWS, no log_metric call).
compile(entry_point_source, "vqe_monitored_job.py", "exec")
print("Wrote vqe_monitored_job.py and confirmed it compiles.")

## 6. Creating the job (gated -- never runs without an explicit opt-in)

`AwsQuantumJob.create(...)` is the one call that hits AWS: it provisions an instance, runs the entry point, and bills you. It stays behind `RUN_ON_AWS = False`.

**COST NOTE.** A Hybrid Job bills two streams added together: the classical instance (~$0.10-$3.00/hour for as long as the container runs) plus the quantum tasks at standard per-task and per-shot rates. This demo runs the inner loop on `LocalSimulator` even inside the job, so the only charge would be the `ml.m5.large` instance for the few seconds it runs. Per the project cost rule, leave `RUN_ON_AWS = False` unless you have explicitly decided to incur that charge.

In [ ]:
RUN_ON_AWS = False  # leave False; flip only with an explicit cost decision

if RUN_ON_AWS:
    # COST NOTE: provisions an ml.m5.large instance and bills until completion.
    job = AwsQuantumJob.create(
        device="local:braket/braket.devices.LocalSimulator",
        source_module="vqe_monitored_job.py",
        entry_point="vqe_monitored_job:main",
        job_name="vqe-monitored-demo",
        hyperparameters={"HP_lr": "0.5", "HP_max_iter": "40", "HP_tol": "1e-4"},
        instance_config={"instanceType": "ml.m5.large", "instanceCount": 1},
        # stopping_condition caps wall-clock as a hard backstop to the tol check:
        stopping_condition={"maxRuntimeInSeconds": 600},
        wait_until_complete=False,
    )
    print("submitted:", job.arn)

    # These also hit AWS -- guarded for the same reason:
    print("state:", job.state())          # QUEUED / RUNNING / COMPLETED
    print("metrics:", job.metrics())       # the 'energy' series, as a DataFrame
    print("result:", job.result())         # what save_job_result wrote
else:
    print("RUN_ON_AWS is False -- skipping AwsQuantumJob.create (no AWS, no cost).")
    print("The convergence curve above is the same series job.metrics() would return.")

## Exercises

Four exercises on reading a job's live signals. Each has tiered hints -- expand
only what you need -- and a check cell that confirms the property once you have
it. Work them locally first; only flip `RUN_ON_AWS` once an exercise truly needs
the service and you accept the cost.

### Exercise 1 — A second metric: the gradient norm

Alongside `energy`, a monitored job usually logs the gradient norm, so CloudWatch
shows not just *where* the optimizer is but *how big a step it is still taking*.
That norm collapses toward zero as the loop nears the minimum.

Re-run the local descent and record the L2 norm of the parameter-shift gradient
at each iteration.

Define `grad_norm_history`: a list holding `float(np.linalg.norm(grad))` for each
iteration of the loop, in order.

<details><summary>Hint 1 — nudge</summary>

The loop in Section 2 already computes `grad` every iteration to update the
parameters -- you just never kept its size. As the optimizer approaches a
minimum, what happens to the magnitude of the gradient?

</details>
<details><summary>Hint 2 — approach</summary>

Reuse the Section 2 loop. Inside it, right after the parameter-shift block fills
`grad`, append `float(np.linalg.norm(grad))` to `grad_norm_history`. In the job
entry-point the analogue is a second `log_metric(metric_name="grad_norm", ...)`
call sitting beside the `energy` one.

</details>

In [ ]:
# Exercise 1: Log the gradient norm alongside the energy each iteration.
# Define: grad_norm_history -- list of float(np.linalg.norm(grad)) per iteration.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert len(grad_norm_history) >= 2, "record one gradient norm per iteration"
    assert all(g >= 0 for g in grad_norm_history), "an L2 norm is never negative"
    assert grad_norm_history[-1] < 0.25 * grad_norm_history[0], (
        "the gradient should shrink markedly as the loop nears the minimum"
    )

### Exercise 2 — Tune the stopping condition

The entry-point breaks early once the energy improves by less than `tol`. A
tighter `tol` buys accuracy but runs longer (and costs more); a looser one stops
sooner but leaves energy on the table. Quantify that trade-off.

Define `run`: a function `run(tol)` that runs the local descent with that
early-stop tolerance and returns `(n_iters, final_energy)` -- the iteration it
stopped at and the energy there. Define `stop_iters`: a dict mapping each `tol`
in `{1e-2, 1e-3, 1e-4}` to the `n_iters` it took.

<details><summary>Hint 1 — nudge</summary>

The stopping rule is the one in the Section 5 entry-point: keep the previous
energy and break when `abs(prev - e) < tol`. A smaller `tol` is a stricter bar to
clear -- so does it stop earlier or later?

</details>
<details><summary>Hint 2 — approach</summary>

Wrap the Section 2 loop in `def run(tol):`. Track `prev` across iterations and
`break` -- returning the current iteration index and energy -- when
`abs(prev - e) < tol`; if it never triggers, return the final iteration and
energy. Then build `stop_iters = {tol: run(tol)[0] for tol in (1e-2, 1e-3, 1e-4)}`.

</details>

In [ ]:
# Exercise 2: Measure how the early-stop tolerance trades iterations for accuracy.
# Define: run(tol) -> (n_iters, final_energy); stop_iters -- {tol: n_iters}.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert callable(run), "run should be a function you can call with a tolerance"
    assert set(stop_iters) >= {1e-2, 1e-3, 1e-4}, "record the three tolerances from the prompt"
    assert stop_iters[1e-2] <= stop_iters[1e-3] <= stop_iters[1e-4], (
        "a tighter tolerance is a stricter bar, so it should run at least as long"
    )
    _n, _e = run(1e-4)
    assert _n >= 1 and _e < exact_min + 0.02, (
        "the tightest tolerance should converge close to the exact minimum"
    )
    assert run(1e-2)[1] > run(1e-4)[1], (
        "a looser tolerance stops sooner and leaves more energy on the table"
    )

### Exercise 3 — Learning-rate sweep

`lr` is a hyperparameter -- in a job it arrives through the `HP_lr` channel. Too
small and the descent crawls; too large and it can overshoot the minimum. Sweep
it locally and compare the convergence curves.

Define `lr_histories`: a dict mapping each learning rate in `{0.1, 0.3, 0.5, 0.8}`
to its energy history (the list of energies across the iterations of a run at
that rate). Overlay the four curves on one plot if you like the picture.

<details><summary>Hint 1 — nudge</summary>

Every run is the same Section 2 loop with a different `lr`. Which rate reaches
the dashed exact-minimum line in the fewest iterations, and does the largest
rate ever step past it?

</details>
<details><summary>Hint 2 — approach</summary>

Loop over the four rates. For each, run the Section 2 descent from `start` with
that `lr` and collect the per-iteration energies into a list; store the list
under the rate in `lr_histories`. Plot each series against its iteration index
on shared axes.

</details>

In [ ]:
# Exercise 3: Sweep the learning rate and compare convergence curves.
# Define: lr_histories -- {lr: [energy per iteration]} for lr in {0.1, 0.3, 0.5, 0.8}.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert set(lr_histories) == {0.1, 0.3, 0.5, 0.8}, "sweep exactly the four rates from the prompt"
    assert all(len(h) >= 2 for h in lr_histories.values()), (
        "each history holds one energy per iteration"
    )
    assert all(h[-1] < h[0] for h in lr_histories.values()), "every rate should drive the energy down"
    assert all(h[-1] < exact_min + 0.2 for h in lr_histories.values()), (
        "each run should land near the exact minimum"
    )
    assert lr_histories[0.8][-1] <= lr_histories[0.1][-1], (
        "the larger rate should converge at least as low within the iteration budget"
    )

### Exercise 4 — The metrics stream is the local curve

`job.metrics()` returns exactly the numbers your entry-point streamed with
`log_metric` -- one row per call, each tagged with its `metric_name`, `value`,
and `iteration_number`. Before spending anything, build that stream from the run
you already have, so you know precisely what the job will report.

Define `metrics_records`: a list of dicts, one per iteration, each of the form
`{"iteration_number": i, "metric_name": "energy", "value": e}`, reproducing the
`energy` series the entry-point would emit from `energy_history`.

<details><summary>Hint 1 — nudge</summary>

Look at the `log_metric(metric_name="energy", value=e, iteration_number=i)` call
inside the Section 5 entry-point. Each call is one row of what `job.metrics()`
later returns. What does the full stream look like across the whole run?

</details>
<details><summary>Hint 2 — approach</summary>

Enumerate `energy_history`. For each `(i, e)`, append
`{"iteration_number": i, "metric_name": "energy", "value": float(e)}` to
`metrics_records`. To see the real thing (this one incurs cost), flip
`RUN_ON_AWS = True`, submit, wait for `COMPLETED`, and compare `job.metrics()`
against this table.

</details>

In [ ]:
# Exercise 4: Reconstruct the 'energy' metric stream the job would emit.
# Define: metrics_records -- [{'iteration_number': i, 'metric_name': 'energy',
#         'value': e}, ...] built from energy_history.

# TODO: your code here

In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    assert len(metrics_records) == len(energy_history), "one record per logged iteration"
    assert all(
        {"iteration_number", "metric_name", "value"} <= set(r) for r in metrics_records
    ), "each record needs iteration_number, metric_name, and value"
    assert all(r["metric_name"] == "energy" for r in metrics_records), (
        "this stream logs the 'energy' metric"
    )
    assert [r["iteration_number"] for r in metrics_records] == list(range(len(energy_history))), (
        "iteration_number should count 0, 1, 2, ... in order"
    )
    assert all(abs(r["value"] - e) < 1e-9 for r, e in zip(metrics_records, energy_history)), (
        "each logged value should reproduce the local energy at that iteration"
    )

## Summary

- A Hybrid Job is a black box until it reports; `log_metric(metric_name, value, iteration_number)` is how the script narrates its own progress to CloudWatch.
- We ran the real VQE loop on `LocalSimulator` (exact expectation, no credentials), collected the energy each iteration, and plotted the descent toward $-\sqrt{1.25}$ -- the same curve CloudWatch streams.
- The three channels: **hyperparameters** in, **input data** staged from S3, **metrics + results** out.
- `log_metric` and `save_job_result` only work inside a running job, so they live solely in the entry-point string; the `AwsQuantumJob.create` call stays behind `RUN_ON_AWS = False` with a cost note.
- A `stopping_condition` (a `tol` check in the loop, plus `maxRuntimeInSeconds` as a backstop) halts a converged job promptly -- the cheapest job is the one that shuts down on time.

**Next:** [`04-checkpointing.ipynb`](04-checkpointing.ipynb) -- make a long-running job survive failure by saving and resuming optimizer state.